# Cross-validation and tuning

The impact estimator is validated with an expanding-window time-series cross-validation: five folds, each trained on every month before its validation month (May to September 2025). Rows are never shuffled and the October–December test months are never used here. The same folds tune the regularisation strength C of the logistic regression and compare it with gradient boosting and the type-and-location baseline. PR-AUC is used because high-impact cases are about 16% of the records.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline

from ml.data import load_frame
from ml.evaluation import best_f1_threshold, classification_metrics, split_dates, temporal_masks
from ml.features import impact_preprocessor, report_time_features
from ml.impact import GRADIENT_BOOSTING, LOGISTIC_REGRESSION, TypeLocationBaseline, fit_pipeline

frame = load_frame(DATA / "ungrd_emergencies_v2.csv", DATA / "divipola_dane.csv")
month = pd.to_datetime(frame["occurred_on"]).dt.month
labels = frame["is_high_impact"].to_numpy(dtype=bool)

## Expanding-window folds

For each validation month, every candidate is fitted on all earlier months and scored with PR-AUC on that month.

In [ ]:
C_VALUES = [0.1, 0.3, 1.0, 3.0]


def logistic(c):
    return Pipeline([
        ("features", impact_preprocessor()),
        ("model", LogisticRegression(C=c, class_weight="balanced", max_iter=2000)),
    ])


rows = []
for validation_month in range(5, 10):
    train = (month < validation_month).to_numpy()
    validation = (month == validation_month).to_numpy()
    X_train = report_time_features(frame[train])
    X_val = report_time_features(frame[validation])
    row = {"month": validation_month, "train_rows": int(train.sum()), "validation_rows": int(validation.sum())}
    for c in C_VALUES:
        scores = logistic(c).fit(X_train, labels[train]).predict_proba(X_val)[:, 1]
        row[f"logistic_C{c}"] = average_precision_score(labels[validation], scores)
    boosting = fit_pipeline(GRADIENT_BOOSTING, frame[train])
    row["gradient_boosting"] = average_precision_score(labels[validation], boosting.predict_proba(X_val)[:, 1])
    baseline = TypeLocationBaseline().fit(frame[train])
    row["baseline"] = average_precision_score(labels[validation], baseline.score(frame[validation]))
    rows.append(row)

folds = pd.DataFrame(rows).set_index("month")
folds.round(3)

In [ ]:
scores = folds.drop(columns=["train_rows", "validation_rows"])
pd.DataFrame({"mean": scores.mean(), "std": scores.std()}).round(3)

## Overfitting check

The selected logistic regression and gradient boosting are fitted on January–September with the threshold tuned on September, then scored on their own training rows and on the October–December test rows.

In [ ]:
masks = temporal_masks(frame, split_dates(frame["occurred_on"]))
fit, validation, train, test = (masks[k].to_numpy() for k in ("fit", "validation", "train", "test"))

results = {}
for kind in (LOGISTIC_REGRESSION, GRADIENT_BOOSTING):
    tuned = fit_pipeline(kind, frame[fit])
    threshold = best_f1_threshold(labels[validation], tuned.predict_proba(report_time_features(frame[validation]))[:, 1])
    model = fit_pipeline(kind, frame[train])
    for part, mask in (("train", train), ("test", test)):
        probability = model.predict_proba(report_time_features(frame[mask]))[:, 1]
        results[(kind, part)] = {"threshold": threshold, **classification_metrics(labels[mask], probability, threshold)}

pd.DataFrame(results).T.round(3)

## Recall by event type on the test set

Shows where the selected logistic regression misses high-impact emergencies.

In [ ]:
tuned = fit_pipeline(LOGISTIC_REGRESSION, frame[fit])
threshold = best_f1_threshold(labels[validation], tuned.predict_proba(report_time_features(frame[validation]))[:, 1])
model = fit_pipeline(LOGISTIC_REGRESSION, frame[train])
test_frame = frame[test].assign(
    flagged=model.predict_proba(report_time_features(frame[test]))[:, 1] >= threshold,
    actual=labels[test],
)
by_type = test_frame.groupby("event_type").apply(
    lambda group: pd.Series({
        "rows": len(group),
        "high_impact": int(group["actual"].sum()),
        "recall": (group["flagged"] & group["actual"]).sum() / max(group["actual"].sum(), 1),
        "share_flagged": group["flagged"].mean(),
    })
)
by_type.sort_values("rows", ascending=False).head(6).round(2)

## Target thresholds from training months only

The high-impact label uses per-type 90th percentiles of people affected and hectares, computed over the whole year. This check recomputes them with January–September only and retrains the impact estimator, to measure how much the test months influence the label.

In [ ]:
from ml.impact import train_impact


def percentile_90(data, column, reference):
    positive = reference[reference[column] > 0]
    return data["event_type"].map(positive.groupby("event_type")[column].quantile(0.9))


reference = frame[pd.to_datetime(frame["occurred_on"]) < pd.Timestamp(split_dates(frame["occurred_on"]).test_start)]
victims = frame["deceased"] + frame["injured"] + frame["missing"] > 0
people = (frame["people_affected"] > 0) & (frame["people_affected"] >= percentile_90(frame, "people_affected", reference))
hectares = (frame["hectares"] > 0) & (frame["hectares"] >= percentile_90(frame, "hectares", reference))
train_only = frame.assign(is_high_impact=victims | (frame["homes_destroyed"] > 0) | people | hectares)

print(f"Labels that change: {int((train_only['is_high_impact'] != frame['is_high_impact']).sum())} of {len(frame):,}")
keys = ["model", "f1", "recall", "precision", "pr_auc", "baseline_f1", "baseline_pr_auc"]
pd.DataFrame({
    "whole year": {k: train_impact(frame).metrics[k] for k in keys},
    "training months": {k: train_impact(train_only).metrics[k] for k in keys},
})